# 04 — Evaluation

Evaluate model performance with metrics, confusion matrix, ROC curve, Precision-Recall curve, hyperparameter tuning, and SHAP explainability.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import joblib
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    roc_curve, precision_recall_curve
)

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [2]:
best_model = joblib.load("../models/best_model.pkl")

processed = joblib.load("../models/processed_data.pkl")

X_test = processed["X_test"]
y_test = processed["y_test"]

print("Model and data loaded successfully")
print("X_test shape:", X_test.shape)

Model and data loaded successfully
X_test shape: (2000, 3)


In [3]:
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

In [4]:
print("Accuracy  :", round(accuracy_score(y_test, y_pred), 4))
print("Precision :", round(precision_score(y_test, y_pred, zero_division=0), 4))
print("Recall    :", round(recall_score(y_test, y_pred, zero_division=0), 4))
print("F1 Score  :", round(f1_score(y_test, y_pred, zero_division=0), 4))
print("ROC-AUC   :", round(roc_auc_score(y_test, y_prob), 4))

Accuracy  : 0.8565
Precision : 0.1746
Recall    : 0.8806
F1 Score  : 0.2914
ROC-AUC   : 0.9488


In [5]:
print(classification_report(y_test, y_pred, target_names=["No Default", "Default"]))

              precision    recall  f1-score   support

  No Default       1.00      0.86      0.92      1933
     Default       0.17      0.88      0.29        67

    accuracy                           0.86      2000
   macro avg       0.58      0.87      0.61      2000
weighted avg       0.97      0.86      0.90      2000



In [6]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.savefig("../data/processed/eval_confusion_matrix.png", dpi=80)
plt.show()
print("Plot saved.")

Plot saved.


In [7]:
fpr, tpr, _ = roc_curve(y_test, y_prob)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f"ROC AUC = {round(roc_auc_score(y_test, y_prob), 4)}")
plt.plot([0, 1], [0, 1], "--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.tight_layout()
plt.savefig("../data/processed/roc_curve.png", dpi=80)
plt.show()
print("Plot saved.")

Plot saved.


In [8]:
precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_prob)

plt.figure(figsize=(8, 6))
plt.plot(recall_vals, precision_vals)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.tight_layout()
plt.savefig("../data/processed/precision_recall_curve.png", dpi=80)
plt.show()
print("Plot saved.")

Plot saved.


In [9]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

param_grid = {
    "n_estimators":    [100, 200, 300],
    "max_depth":       [5, 10, 15, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf":  [1, 2, 4]
}

rf = RandomForestClassifier(random_state=42, n_jobs=-1)

search = RandomizedSearchCV(
    rf,
    param_grid,
    n_iter=10,
    cv=3,
    scoring="roc_auc",
    random_state=42,
    n_jobs=-1
)

X_train_smote = processed["X_train_smote"]
y_train_smote = processed["y_train_smote"]

search.fit(X_train_smote, y_train_smote)

print("Best Params:", search.best_params_)
print("Best CV ROC-AUC:", round(search.best_score_, 4))

Best Params: {'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_depth': 15}
Best CV ROC-AUC: 0.9722


In [10]:
# Save the tuned model as best_model (overwrites previous)
joblib.dump(search.best_estimator_, "../models/best_model.pkl")

print("Tuned Final Model Saved")

Tuned Final Model Saved


In [11]:
import shap

# Use a sample of 200 rows for speed
X_sample = X_test[:200]

explainer   = shap.TreeExplainer(search.best_estimator_)
shap_values = explainer.shap_values(X_sample)

# For binary classifiers shap_values is a list [class0, class1]
sv = shap_values[1] if isinstance(shap_values, list) else shap_values

preprocessor   = joblib.load("../models/preprocessor.pkl")
feature_names  = preprocessor.get_feature_names_out()

shap.summary_plot(sv, X_sample, feature_names=feature_names, show=False)
plt.tight_layout()
plt.savefig("../data/processed/shap_summary.png", dpi=80, bbox_inches="tight")
plt.show()
print("SHAP plot saved.")

SHAP plot saved.


In [12]:
sample = X_test[0:1]

prediction  = search.best_estimator_.predict(sample)
probability = search.best_estimator_.predict_proba(sample)

print("Prediction  :", prediction)
print("Probability :", probability)

Prediction  : [0]
Probability : [[0.97865691 0.02134309]]


In [13]:
print("=" * 60)
print("PROJECT COMPLETED SUCCESSFULLY")
print("=" * 60)
print("""
✔ Data Cleaning
✔ EDA
✔ Feature Engineering
✔ Encoding & Scaling
✔ Train/Test Split
✔ SMOTE Oversampling
✔ Model Training (9 algorithms)
✔ Model Comparison
✔ Hyperparameter Tuning
✔ SHAP Explainability
✔ Model Saving
✔ Ready for Streamlit Deployment
""")

PROJECT COMPLETED SUCCESSFULLY

✔ Data Cleaning
✔ EDA
✔ Feature Engineering
✔ Encoding & Scaling
✔ Train/Test Split
✔ SMOTE Oversampling
✔ Model Training (9 algorithms)
✔ Model Comparison
✔ Hyperparameter Tuning
✔ SHAP Explainability
✔ Model Saving
✔ Ready for Streamlit Deployment

